## a) Definirea modelului

Stările ascunse sunt dificultățile testelor, în ordinea **D, M, U** (dificil, mediu, ușor). Observațiile sunt notele, în ordinea **FB, B, S, NS**.

Prima dificultate poate fi oricare dintre cele trei, cu aceeași probabilitate:

$$
\pi=\begin{pmatrix}1/3&1/3&1/3\end{pmatrix}.
$$

Matricea de tranziție este:

$$
A=\begin{pmatrix}
0&0.5&0.5\\
0.5&0.25&0.25\\
0.5&0.25&0.25
\end{pmatrix}.
$$

Elementul $a_{ij}$ este probabilitatea de a trece de la dificultatea $i$ la dificultatea $j$. Rândurile și coloanele sunt în ordinea D, M, U.

Matricea de emisie este:

$$
B=\begin{pmatrix}
0.10&0.20&0.40&0.30\\
0.15&0.25&0.50&0.10\\
0.20&0.30&0.40&0.10
\end{pmatrix}.
$$

Rândurile corespund dificultăților D, M, U, iar coloanele notelor FB, B, S, NS. Elementul $b_j(o_t)$ este probabilitatea notei $o_t$ dacă dificultatea este $j$.

Pentru $A$, sumele rândurilor sunt $0+0.5+0.5=1$, respectiv $0.5+0.25+0.25=1$ pentru ultimele două rânduri.

Pentru $B$, obțin:

$$
0.10+0.20+0.40+0.30=1,
$$

$$
0.15+0.25+0.50+0.10=1,
$$

$$
0.20+0.30+0.40+0.10=1.
$$

Și suma elementelor lui $\pi$ este 1.

In [30]:
import numpy as np

stari = ["D", "M", "U"]
note_posibile = ["FB", "B", "S", "NS"]

pi = np.array([1/3, 1/3, 1/3])

A = np.array([
    [0,   0.5,  0.5],
    [0.5, 0.25, 0.25],
    [0.5, 0.25, 0.25]
])

B = np.array([
    [0.10, 0.20, 0.40, 0.30],
    [0.15, 0.25, 0.50, 0.10],
    [0.20, 0.30, 0.40, 0.10]
])

note = ["FB", "FB", "S", "B", "B", "S",
        "B", "B", "NS", "B", "B", "S"]

observatii = [note_posibile.index(nota) for nota in note]

print("Suma lui pi:", pi.sum())
print("Sumele rândurilor din A:", A.sum(axis=1))
print("Sumele rândurilor din B:", B.sum(axis=1))

Suma lui pi: 1.0
Sumele rândurilor din A: [1. 1. 1.]
Sumele rândurilor din B: [1. 1. 1.]


## b) Inițializarea Viterbi

Notez cu $\delta_t(j)$ probabilitatea comună maximă a unui traseu de dificultăți care se termină în starea $j$ la momentul $t$, împreună cu primele $t$ note observate.

Prima notă este FB. Aplic formula $\delta_1(j)=\pi_jb_j(FB)$:

$$
\delta_1(D)=0.3333\cdot0.10=0.0333
$$

$$
\delta_1(M)=0.3333\cdot0.15=0.0500
$$

$$
\delta_1(U)=0.3333\cdot0.20=0.0667
$$

In [31]:
T = len(observatii)
N = len(stari)

delta = []
psi = []

for i in range(T):
    delta.append([0] * N)
    psi.append([-1] * N)

prima_nota = observatii[0]

for j in range(N):
    probabilitate_initiala = pi[j]
    probabilitate_nota = B[j][prima_nota]

    delta[0][j] = probabilitate_initiala * probabilitate_nota

print("Valorile delta_1:")

for j in range(N):
    print(stari[j], ":", f"{delta[0][j]:.4f}")

Valorile delta_1:
D : 0.0333
M : 0.0500
U : 0.0667


## c) Implementarea recurenței

Pentru fiecare moment și fiecare stare $j$, calculez scorurile pentru toate stările precedente $i$. Aleg scorul maxim și îl înmulțesc cu probabilitatea notei curente.

$$
\delta_t(j)=b_j(o_t)\max_i\{\delta_{t-1}(i)a_{ij}\}.
$$

În al doilea tabel rețin indicele stării precedente care a dat maximul.

$$
\psi_t(j)=\arg\max_i\{\delta_{t-1}(i)a_{ij}\}.
$$

In [32]:
for t in range(1, T):
    for j in range(N):
        scor_maxim = -1

        for i in range(N):
            scor = delta[t - 1][i] * A[i][j]

            if scor > scor_maxim:
                scor_maxim = scor
                psi[t][j] = i

        nota_curenta = observatii[t]
        delta[t][j] = scor_maxim * B[j][nota_curenta]

## d) Verificarea intermediară

$$
\begin{aligned}
\delta_2(D)
&= 0.10 \cdot \max\{
0.0333 \cdot 0,\;
0.05 \cdot 0.5,\;
0.0666 \cdot 0.5
\}\\
&= 0.10 \cdot \max\{
0,\; 0.025,\; 0.0333
\}\\
&= 0.10 \cdot 0.0333\\
&= \boxed{0.00333}.
\end{aligned}
$$

$$
\begin{aligned}
\delta_2(M)
&= 0.15 \cdot \max\{
0.0333 \cdot 0.5,\;
0.05 \cdot 0.25,\;
0.0666 \cdot 0.25
\}\\
&= 0.15 \cdot \max\{
0.01665,\; 0.0125,\; 0.01665
\}\\
&= 0.15 \cdot 0.01665\\
&= \boxed{0.0024975}.
\end{aligned}
$$

$$
\begin{aligned}
\delta_2(U)
&= 0.20 \cdot \max\{
0.0333 \cdot 0.5,\;
0.05 \cdot 0.25,\;
0.0666 \cdot 0.25
\}\\
&= 0.20 \cdot \max\{
0.01665,\; 0.0125,\; 0.01665
\}\\
&= 0.20 \cdot 0.01665\\
&= \boxed{0.00333}.
\end{aligned}
$$

In [33]:
print("           D       M       U")

for t in range(2):
    valori = "  ".join(f"{valoare:.4f}" for valoare in delta[t])
    print(f"delta_{t + 1}: {valori}")

           D       M       U
delta_1: 0.0333  0.0500  0.0667
delta_2: 0.0033  0.0025  0.0033


## e) Reconstrucția traseului

Aleg starea cu scor maxim din ultimul rând al lui `delta`. Apoi, parcurg momentele în ordine inversă, folosind indicii din `psi` pentru a găsi stările precedente.

Pot exista mai multe trasee cu același scor maxim. Codul întoarce unul dintre ele, folosind primul indice în caz de egalitate.

In [34]:
traseu = [0] * T

stare_finala = 0

for j in range(N):
    if delta[T - 1][j] > delta[T - 1][stare_finala]:
        stare_finala = j

traseu[T - 1] = stare_finala
prob_comuna = delta[T - 1][stare_finala]

for t in range(T - 1, 0, -1):
    stare_curenta = traseu[t]
    traseu[t - 1] = psi[t][stare_curenta]

dificultati = []

for i in traseu:
    dificultati.append(stari[i])

print("Secvența de dificultăți:", dificultati)
print("Probabilitatea comună maximă:", prob_comuna)
print()

print("Test / Notă / Dificultate")

for t in range(T):
    print(t + 1, note[t], dificultati[t])

Secvența de dificultăți: ['D', 'U', 'D', 'U', 'D', 'M', 'D', 'U', 'D', 'U', 'D', 'M']
Probabilitatea comună maximă: 2.1093750000000005e-11

Test / Notă / Dificultate
1 FB D
2 FB U
3 S D
4 B U
5 B D
6 S M
7 B D
8 B U
9 NS D
10 B U
11 B D
12 S M


## f) Interpretare

Algoritmul Viterbi caută secvența completă de dificultăți cu probabilitate maximă. El ține cont atât de probabilitatea notelor pentru fiecare dificultate, cât și de tranzițiile dintre dificultăți. Alegerea separată a celei mai probabile dificultăți pentru fiecare test poate produce un traseu mai puțin probabil sau chiar imposibil. De exemplu, în acest model nu se poate trece din D direct în D, deoarece $a_{DD}=0$.

Scorul calculat la e) este probabilitatea comună $P(x^*,o)$ a traseului și a notelor observate. Nu este probabilitatea condiționată $P(x^*\mid o)$.
